
---
# 1. **Imports & Enums** (Setup)
---


In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from enum import Enum
from typing import List, Union, Optional, Tuple
from dataclasses import dataclass


In [2]:

# ==========================================
# 0. CONSTANTS & ENUMS
# ==========================================

class AtomMode(Enum):
    FILTER = "FILTER"    # Local: Learned Compression + Signal Mixing
    ROUTING = "ROUTING"  # Global: Query Bank Scanning (Attention)

class CompoundMode(Enum):
    SERIES = "SERIES"    # Sequential (Layer 1 -> Layer 2)
    PARALLEL = "PARALLEL" # Concurrent (Branch A + Branch B)

class CompoundOp(Enum):
    PRODUCT = "PRODUCT" # Interaction (Modulation)
    SUM = "SUM"         # Superposition (Weighted Add)
    CONCAT = "CONCAT"   # Channel Expansion



---
# 2. **`GraphConfig` Dataclass** (Global settings)
---

In [3]:
@dataclass
class GraphConfig:
    """
    Global Hyperparameters.
    """
    basis_degree: int = 15        # N: Resolution of every curve
    input_channels: int = 1       # 1 for Grayscale, 3 for RGB
    min_uncertainty: float = 1e-6 # Epsilon for TrustGate
    device: str = 'cpu'           # <--- ADDED THIS FIELD

    @property
    def n_coeffs(self) -> int:
        return self.basis_degree + 1

# Global Instance
G_CONFIG = GraphConfig()

# 3. **`SpectralAlgebra`** (The Math Kernels - *stateless*)

In [4]:
import torch
import torch.nn.functional as F
import numpy as np
from typing import Tuple

class SpectralAlgebra:
    """
    The Laws of the Universe (Production Logic).
    Stateless Chebyshev arithmetic operations.
    Updated to handle arbitrary input shapes (Batch, Seq, N).
    """

    # Cache for transform matrices
    _dct_mat = None
    _idct_mat = None
    _grid_points = None

    @classmethod
    def _init_basis(cls, N: int, device: torch.device):
        """Pre-computes DCT matrices for fast spectral transforms."""
        if cls._dct_mat is not None and cls._dct_mat.shape[0] == N and cls._dct_mat.device == device:
            return

        # 1. Chebyshev Grid
        n = torch.arange(N, device=device).float()
        grid = torch.cos(np.pi * n / (N - 1))

        # 2. IDCT Matrix (Coeffs -> Values)
        k = n.view(-1, 1)
        n_idx = n.view(1, -1)
        idct_mat = torch.cos(k * n_idx * np.pi / (N - 1))

        # 3. DCT Matrix (Values -> Coeffs)
        dct_mat = torch.linalg.pinv(idct_mat)

        cls._dct_mat = dct_mat
        cls._idct_mat = idct_mat
        cls._grid_points = grid

    @staticmethod
    def mix(signal_a: torch.Tensor, signal_b: torch.Tensor) -> torch.Tensor:
        """
        Interaction: Multiplies two signals.
        Input: (..., N) arbitrary leading dims.
        """
        # 1. Capture Shape
        # We treat everything before the last dim as 'Batch'
        shape_a = signal_a.shape
        N = shape_a[-1]
        device = signal_a.device

        # 2. Flatten inputs to (Total_Batch, N)
        flat_a = signal_a.reshape(-1, N)
        flat_b = signal_b.reshape(-1, N)

        # 3. Initialize Basis
        SpectralAlgebra._init_basis(N, device)

        # 4. Transform to Values (Physical Space)
        # (Total_Batch, N) @ (N, N) -> (Total_Batch, N)
        val_a = flat_a @ SpectralAlgebra._idct_mat.T
        val_b = flat_b @ SpectralAlgebra._idct_mat.T

        # 5. Interaction
        val_out = val_a * val_b

        # 6. Transform back
        coeffs_out = val_out @ SpectralAlgebra._dct_mat.T

        # 7. Reshape to original dimensions
        return coeffs_out.view(shape_a)

    @staticmethod
    def measure(query: torch.Tensor, target: torch.Tensor) -> torch.Tensor:
        """Similarity: Weighted Inner Product."""
        N = query.shape[-1]
        weights = torch.ones(N, device=query.device) * (np.pi / 2)
        weights[0] = np.pi

        # Weighted Sum
        return (query * target * weights).sum(dim=-1, keepdim=True)

    @staticmethod
    def domain_map(coeffs: torch.Tensor,
                   src_domain: Tuple[float, float],
                   target_domain: Tuple[float, float]) -> torch.Tensor:
        """Relativity: Resamples the curve."""
        if src_domain == target_domain:
            return coeffs

        shape = coeffs.shape
        N = shape[-1]
        device = coeffs.device

        # Flatten
        flat_coeffs = coeffs.reshape(-1, N)

        SpectralAlgebra._init_basis(N, device)

        # ... (Same Matrix construction logic as before) ...
        # For brevity, reusing the core logic logic:

        # Target Grid -> Physical t
        target_grid = SpectralAlgebra._grid_points
        t_scale = 0.5 * (target_domain[1] - target_domain[0])
        t_bias  = 0.5 * (target_domain[1] + target_domain[0])
        t_points = t_scale * target_grid + t_bias

        # Physical t -> Source x
        src_denom = src_domain[1] - src_domain[0]
        x_mapped = (2.0 * t_points - (src_domain[1] + src_domain[0])) / src_denom

        # Evaluation Matrix
        T_mat = torch.zeros(N, N, device=device)
        T_mat[0, :] = 1.0
        T_mat[1, :] = x_mapped
        for n in range(2, N):
            T_mat[n, :] = 2 * x_mapped * T_mat[n-1, :] - T_mat[n-2, :]

        # Execute Map
        new_values = flat_coeffs @ T_mat
        new_coeffs = new_values @ SpectralAlgebra._dct_mat.T

        # Un-Flatten
        return new_coeffs.view(shape)

    @staticmethod
    def energy(coeffs: torch.Tensor) -> torch.Tensor:
        c0 = coeffs[..., 0]
        ck = coeffs[..., 1:]
        return (c0**2 + 0.5 * torch.sum(ck**2, dim=-1)).unsqueeze(-1)

    @staticmethod
    def compress(coeffs: torch.Tensor, target_degree: int) -> torch.Tensor:
        return coeffs[..., :target_degree+1]

In [5]:
import unittest
import torch
import numpy as np

# ==========================================
# 1. THE CLASS UNDER TEST (Paste Production Class Here)
# ==========================================
# (Assuming the SpectralAlgebra class defined above is available in scope)

# ==========================================
# 2. THE TEST SUITE
# ==========================================

class TestSpectralPhysicsProduction(unittest.TestCase):

    def setUp(self):
        # Configuration
        self.device = 'cpu' # Use 'cuda' if available
        self.B = 2          # Batch Size
        self.N = 8          # Degree (Power of 2 is nice for FFT/DCT concepts, though matrix works for any)

        # Initialize Basis Trigger
        SpectralAlgebra._init_basis(self.N, torch.device(self.device))

        # Create Standard Basis Vectors
        # T0 (Constant 1)
        self.t0 = torch.zeros(self.B, self.N, device=self.device)
        self.t0[:, 0] = 1.0

        # T1 (Line x)
        self.t1 = torch.zeros(self.B, self.N, device=self.device)
        self.t1[:, 1] = 1.0

        # T2 (Parabola 2x^2 - 1)
        self.t2 = torch.zeros(self.B, self.N, device=self.device)
        self.t2[:, 2] = 1.0

    def test_mix_interaction(self):
        """
        Verify Pseudo-Spectral Multiplication matches Analytic Identity.
        """
        # 1. Identity: T1 * T0 = T1
        # (This verifies the DCT -> Mult -> IDCT pipeline works for identity)
        res_identity = SpectralAlgebra.mix(self.t1, self.t0)
        self.assertTrue(torch.allclose(res_identity, self.t1, atol=1e-5),
                        f"Identity Mix Failed. Max Error: {(res_identity - self.t1).abs().max()}")

        # 2. Expansion: T1 * T1 = 0.5*T0 + 0.5*T2
        # (This verifies frequency generation works in the Value domain)
        res_square = SpectralAlgebra.mix(self.t1, self.t1)

        expected = torch.zeros_like(res_square)
        expected[:, 0] = 0.5
        expected[:, 2] = 0.5

        self.assertTrue(torch.allclose(res_square, expected, atol=1e-5),
                        f"Square Mix Failed. Max Error: {(res_square - expected).abs().max()}")
        print("Test 1 (Mix): DCT Multiplication Validated")

    def test_domain_map_logic(self):
        """
        Verify Affine Resampling (The hard part).
        """
        # Scenario:
        # Source: A Line y = t on domain [0, 1].
        # Target: We want the slice from [0.0, 0.5].
        #   At t=0.0 (Start), y=0.0
        #   At t=0.5 (End),   y=0.5
        # The new coefficients, when evaluated on the new domain [0, 0.5],
        # must produce these values.

        # 1. Define Source: y = t on [0, 1]
        # In Chebyshev on [0, 1]: y = 0.5*T0 + 0.5*T1
        # (Since T1 spans -1 to 1, we shift/scale)
        src_coeffs = torch.zeros(1, self.N)
        src_coeffs[:, 0] = 0.5
        src_coeffs[:, 1] = 0.5

        src_domain = (0.0, 1.0)
        tgt_domain = (0.0, 0.5)

        # 2. Perform Map
        new_coeffs = SpectralAlgebra.domain_map(src_coeffs, src_domain, tgt_domain)

        # 3. Verify Values at Endpoints
        # We can't easily check coefficients by eye, so we check the VALUES they produce.
        # We define a helper to evaluate Chebyshev coeffs at -1 (Start) and 1 (End)
        def eval_cheb(c, x_val):
            # c: (B, N), x_val: scalar -1 or 1
            # T_n(-1) = (-1)^n, T_n(1) = 1
            val = torch.zeros(c.shape[0])
            for n in range(c.shape[1]):
                basis = (x_val ** n) # -1 or 1 logic works for T_n at endpoints
                val += c[:, n] * basis
            return val

        val_start = eval_cheb(new_coeffs, -1.0) # Should be 0.0 (Start of slice)
        val_end   = eval_cheb(new_coeffs, 1.0)  # Should be 0.5 (End of slice)

        print(f"Test 2 (Domain Map): Start Val={val_start.item():.4f}, End Val={val_end.item():.4f}")

        self.assertTrue(abs(val_start.item() - 0.0) < 1e-4, "Start point mismatch")
        self.assertTrue(abs(val_end.item() - 0.5) < 1e-4, "End point mismatch")

    def test_orthogonality(self):
        """
        Verify Weighted Inner Product.
        """
        # T0 and T1 are orthogonal in Chebyshev space.
        score = SpectralAlgebra.measure(self.t0, self.t1)
        self.assertTrue(score.abs().max() < 1e-5, "Orthogonality check failed")
        print("Test 3 (Measure): Orthogonality Validated")

if __name__ == '__main__':
    # Run tests
    suite = unittest.TestLoader().loadTestsFromTestCase(TestSpectralPhysicsProduction)
    unittest.TextTestRunner(verbosity=0).run(suite)

----------------------------------------------------------------------
Ran 3 tests in 0.006s

OK


Test 2 (Domain Map): Start Val=-0.0000, End Val=0.5000
Test 1 (Mix): DCT Multiplication Validated
Test 3 (Measure): Orthogonality Validated



---
# 4. **`Atom`** (The State)
---

In [6]:
# ==========================================
# 1.1 THE ATOM (State & Logic)
# ==========================================

class MicroCompressor(nn.Module):
    """
    The Neural Reduction Engine.
    Learns to merge two adjacent tokens into one, preserving manifold structure.
    (e.g., merging Sine + Cosine -> Phase Shift).
    """
    def __init__(self, n_coeffs: int):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_coeffs * 2, n_coeffs * 2),
            nn.GELU(),
            nn.Linear(n_coeffs * 2, n_coeffs)
        )

    def forward(self, token_a: torch.Tensor, token_b: torch.Tensor) -> torch.Tensor:
        combined = torch.cat([token_a, token_b], dim=-1)
        return self.net(combined)

class Atom(nn.Module):
    """
    The Fundamental Unit of Processing.
    (Updated to handle Odd Sequence Lengths during Reduction)
    """
    def __init__(self,
                 mode: AtomMode,
                 reduction_factor: int = 1,
                 query_bank_size: int = 64):
        super().__init__()
        self.mode = mode
        self.reduction_factor = reduction_factor
        self.n = G_CONFIG.n_coeffs

        # --- COMPONENT 1: REDUCTION ---
        if self.mode == AtomMode.FILTER and reduction_factor > 1:
            self.compressor = MicroCompressor(self.n)

        # --- COMPONENT 2: MEMORY ---
        if self.mode == AtomMode.FILTER:
            self.primary_weights = nn.Parameter(torch.randn(1, self.n))
        elif self.mode == AtomMode.ROUTING:
            self.primary_weights = nn.Parameter(torch.randn(query_bank_size, self.n))

        # --- COMPONENT 3: ACTIVATION ---
        self.phi = nn.Parameter(torch.ones(self.n))

    def forward(self, stack: torch.Tensor) -> torch.Tensor:
        if self.mode == AtomMode.FILTER:
            return self._forward_filter(stack)
        elif self.mode == AtomMode.ROUTING:
            return self._forward_routing(stack)
        else:
            raise ValueError(f"Unknown Mode: {self.mode}")

    def _forward_filter(self, stack: torch.Tensor) -> torch.Tensor:
        """
        Local Processing Logic with Robust Reduction.
        """
        current_stack = stack

        # 1. Topological Reduction (Micro-Compression)
        if self.reduction_factor > 1:
            B, L, C = current_stack.shape

            # --- FIX: HANDLE ODD LENGTHS ---
            if L % 2 != 0:
                # Pad by duplicating the last token
                # This ensures we have pairs for every token.
                last_token = current_stack[:, -1:, :]
                current_stack = torch.cat([current_stack, last_token], dim=1)
                # New length is L+1 (Even)

            # Reshape to isolate pairs: (Batch, L_new/2, C)
            left = current_stack[:, 0::2, :]
            right = current_stack[:, 1::2, :]

            # Neural Merge
            current_stack = self.compressor(left, right)

        # 2. Analytic Interaction (The Physics)
        projector = self.primary_weights # (1, N)

        B, L, C = current_stack.shape
        flat_stack = current_stack.view(-1, C)
        flat_proj = projector.expand(flat_stack.shape[0], -1)

        mixed = SpectralAlgebra.mix(flat_stack, flat_proj)
        mixed = mixed.view(B, L, C)

        # 3. Activation
        return mixed * self.phi

    def _forward_routing(self, stack: torch.Tensor) -> torch.Tensor:
        """
        Global Processing Logic (Router).
        """
        weights = torch.ones(self.n, device=stack.device) * (np.pi / 2)
        weights[0] = np.pi

        w_stack = stack * weights
        scores = torch.matmul(w_stack, self.primary_weights.T)
        attn = F.softmax(scores, dim=1)

        routed = torch.bmm(attn.transpose(1, 2), stack)
        return routed * self.phi

In [7]:
class TestAtom(unittest.TestCase):
    def setUp(self):
        # Configuration
        G_CONFIG.basis_degree = 7
        self.B = 2
        self.SeqLen = 10
        self.C = G_CONFIG.n_coeffs # 8
        self.device = 'cpu'

        # Dummy Input Stack
        self.stack = torch.randn(self.B, self.SeqLen, self.C)

    def test_filter_reduction(self):
        """Verify MicroCompressor integration."""
        # Create a Reducing Atom (Factor 2)
        atom = Atom(AtomMode.FILTER, reduction_factor=2)

        # Run Forward
        output = atom(self.stack)

        # Check 1: Sequence Length Halved?
        expected_len = self.SeqLen // 2
        self.assertEqual(output.shape[1], expected_len,
                         f"Reduction Failed. Got {output.shape[1]}, expected {expected_len}")

        # Check 2: Signal Changed? (Phi and Mix should alter values)
        self.assertFalse(torch.allclose(output[0,0], self.stack[0,0]),
                         "Signal passed through unchanged!")

        print("Test 1 (Atom Filter): Reduction Logic Validated")

    def test_routing_fixed_output(self):
        """Verify Variable -> Fixed mapping."""
        K = 4 # Bank Size
        atom = Atom(AtomMode.ROUTING, query_bank_size=K)

        # Run Forward
        output = atom(self.stack)

        # Check 1: Output Dimension should be K, not SeqLen
        self.assertEqual(output.shape[1], K,
                         f"Routing Failed. Got len {output.shape[1]}, expected {K}")

        # Check 2: Test with DIFFERENT input size (Variable N)
        stack_large = torch.randn(self.B, 100, self.C) # 100 Tokens
        output_large = atom(stack_large)

        self.assertEqual(output_large.shape[1], K,
                         "Routing failed to handle variable input size.")

        print("Test 2 (Atom Routing): Variable-to-Fixed Mapping Validated")

if __name__ == '__main__':
    suite = unittest.TestLoader().loadTestsFromTestCase(TestAtom)
    unittest.TextTestRunner(verbosity=0).run(suite)

Test 1 (Atom Filter): Reduction Logic Validated


----------------------------------------------------------------------
Ran 2 tests in 0.084s

OK


Test 2 (Atom Routing): Variable-to-Fixed Mapping Validated



---
# 5. **`Compound`** (The Topology)
---

In [8]:
class Compound(nn.Module):
    """
    The Structural Composite.
    Defines how sub-units (Atoms or other Compounds) are connected.

    Modes:
    1. SERIES:   Sequential chain (Layer i -> Layer i+1).
    2. PARALLEL: Concurrent execution (Branch A || Branch B).
                 Requires a 'CompoundOp' to merge results.
    """
    def __init__(self,
                 sub_nodes: List[Union['Compound', Atom]],
                 mode: CompoundMode,
                 op: Optional[CompoundOp] = None):
        super().__init__()
        self.sub_nodes = nn.ModuleList(sub_nodes)
        self.mode = mode
        self.op = op

        # Validation
        if self.mode == CompoundMode.PARALLEL and self.op is None:
            raise ValueError("PARALLEL mode requires a valid CompoundOp (SUM, PRODUCT, CONCAT).")
        if self.mode == CompoundMode.PARALLEL and self.op == CompoundOp.PRODUCT and len(sub_nodes) != 2:
            raise ValueError("PRODUCT op currently supports exactly 2 branches (Subject * Tool).")

    def forward(self, stack: torch.Tensor) -> torch.Tensor:
        # --- MODE 1: SERIES (Sequential) ---
        # Logic: x = f(g(h(x)))
        if self.mode == CompoundMode.SERIES:
            current_signal = stack
            for node in self.sub_nodes:
                current_signal = node(current_signal)
            return current_signal

        # --- MODE 2: PARALLEL (Distributed) ---
        # Logic: y = f(x) op g(x)
        elif self.mode == CompoundMode.PARALLEL:
            # 1. Execute all branches independently
            results = [node(stack) for node in self.sub_nodes]

            # 2. Merge Results
            if self.op == CompoundOp.SUM:
                # Additive Superposition (Interference)
                # Note: In a full system, we would check 'SpectralAlgebra.energy' here
                # for TrustGate weighting. For V1.3, we use direct sum.
                return sum(results)

            elif self.op == CompoundOp.PRODUCT:
                # Receiver-Centric Modulation (Subject * Tool)
                # Delegates to the Physics Engine's 'mix'
                return SpectralAlgebra.mix(results[0], results[1])

            elif self.op == CompoundOp.CONCAT:
                # Channel Expansion
                # Concatenates along the Coefficient/Feature dimension (dim=-1)
                return torch.cat(results, dim=-1)

            else:
                raise ValueError(f"Unknown Op: {self.op}")

In [9]:
import unittest

class TestCompoundFixed(unittest.TestCase):
    def setUp(self):
        G_CONFIG.basis_degree = 7
        self.B = 2
        self.SeqLen = 10
        self.C = G_CONFIG.n_coeffs # 8
        self.stack = torch.randn(self.B, self.SeqLen, self.C)

    def test_parallel_product_fixed(self):
        """Verify Parallel Interaction (Mix(A, B)) with 3D tensors."""
        atom1 = Atom(AtomMode.FILTER)
        atom2 = Atom(AtomMode.FILTER)

        compound = Compound([atom1, atom2], CompoundMode.PARALLEL, CompoundOp.PRODUCT)

        # Manual Product
        out1 = atom1(self.stack)
        out2 = atom2(self.stack)
        expected = SpectralAlgebra.mix(out1, out2)

        # Compound Product
        result = compound(self.stack)

        # Check Shape (Should remain 3D)
        self.assertEqual(result.shape, (self.B, self.SeqLen, self.C),
                         "Output shape mismatch!")

        self.assertTrue(torch.allclose(result, expected),
                        "Parallel Product (Mix) failed.")
        print("Test (Compound Product 3D): Validated")

if __name__ == '__main__':
    suite = unittest.TestLoader().loadTestsFromTestCase(TestCompoundFixed)
    unittest.TextTestRunner(verbosity=0).run(suite)

----------------------------------------------------------------------
Ran 1 test in 0.002s

OK


Test (Compound Product 3D): Validated


# 6. **`Core`**

In [10]:
class Core:
    """
    Session Manager (The Engine Room).
    Handles global settings, device management, and optimization modes.
    """

    @staticmethod
    def set_device(device_name: str):
        """Updates the global physics configuration."""
        if device_name == 'cuda' and not torch.cuda.is_available():
            print("Warning: CUDA requested but not available. Falling back to CPU.")
            device_name = 'cpu'

        G_CONFIG.device = device_name
        # Note: This trigger update allows new Atoms to initialize on the correct device.

    @staticmethod
    def get_device() -> torch.device:
        """Returns the actual torch.device object."""
        return torch.device(G_CONFIG.device)

    @staticmethod
    def update_config(degree: Optional[int] = None, uncertainty_threshold: Optional[float] = None):
        """Runtime updates to physics constants."""
        if degree is not None:
            G_CONFIG.basis_degree = degree
        if uncertainty_threshold is not None:
            G_CONFIG.min_uncertainty = uncertainty_threshold

# 7. **`Module`**

In [11]:
class Module(nn.Module):
    """
    The Black Box API.
    Standard base class for all High-Level Components (Retina, Cortex, System).
    """
    def __init__(self):
        super().__init__()

    @property
    def device(self) -> torch.device:
        """Helper: Get the device this module is living on."""
        # Check first parameter
        try:
            return next(self.parameters()).device
        except StopIteration:
            return torch.device('cpu') # Stateless module

    def count_parameters(self) -> int:
        """Utility: How heavy is this block?"""
        return sum(p.numel() for p in self.parameters() if p.requires_grad)

    def save(self, path: str):
        """Standardized checkpointing."""
        torch.save(self.state_dict(), path)

    def load(self, path: str):
        """Standardized loading."""
        self.load_state_dict(torch.load(path, map_location=self.device))

In [12]:
class TestCoreAndModule(unittest.TestCase):

    def test_core_config(self):
        """Verify Core updates global config."""
        # 1. Change Degree
        Core.update_config(degree=20)
        self.assertEqual(G_CONFIG.basis_degree, 20, "Core failed to update degree")
        self.assertEqual(G_CONFIG.n_coeffs, 21, "Derived property n_coeffs failed")

        # 2. Reset
        Core.update_config(degree=7)
        self.assertEqual(G_CONFIG.basis_degree, 7)
        print("Test 1 (Core): Config Management Validated")

    def test_module_basics(self):
        """Verify Module inheritance and utilities."""
        # Create a dummy module
        class TestBlock(Module):
            def __init__(self):
                super().__init__()
                self.layer = nn.Linear(10, 10)

        block = TestBlock()

        # 1. Check inheritance
        self.assertTrue(isinstance(block, nn.Module), "Module must be a torch.nn.Module")

        # 2. Check Parameter Count (10*10 weights + 10 bias = 110)
        self.assertEqual(block.count_parameters(), 110,
                         f"Param count wrong. Got {block.count_parameters()}, expected 110")

        print("Test 2 (Module): Wrapper Utilities Validated")

if __name__ == '__main__':
    suite = unittest.TestLoader().loadTestsFromTestCase(TestCoreAndModule)
    unittest.TextTestRunner(verbosity=0).run(suite)

----------------------------------------------------------------------
Ran 2 tests in 0.002s

OK


Test 1 (Core): Config Management Validated
Test 2 (Module): Wrapper Utilities Validated


#


---
# 8. **`Interface`** (The Translator)
---

In [13]:
class Interface(Module):
    """
    The Translator.
    Handles Pixels <-> Chebyshev Coefficients.
    Responsible for Tokenization, Topology Generation (Bridges), and Serialization.
    """
    def __init__(self, patch_size: int = 4, image_channels: int = 1):
        super().__init__()
        self.patch_size = patch_size
        self.channels = image_channels
        self.degree = G_CONFIG.basis_degree
        self.n_coeffs = G_CONFIG.n_coeffs

        # 1. The "Fitter" (Pixel -> Spectral Projector)
        # We use a Linear layer to project the raw pixel patch onto the basis.
        # This is mathematically equivalent to a learned basis decomposition.
        input_dim = (patch_size ** 2) * image_channels
        self.projector = nn.Linear(input_dim, self.n_coeffs)

    def input_processor(self, image: torch.Tensor) -> torch.Tensor:
        """
        Raw Image -> Spectral Chain.
        """
        # 1. Patchify (Pixels -> Grid of Patches)
        # Output: (Batch, N_Patches, Patch_Pixels)
        patches = self._patchify(image)

        # 2. Fit Chebyshev (Data Tokens)
        # Project pixels to coefficients
        # Output: (Batch, N_Patches, N_Coeffs)
        data_tokens = self.projector(patches)

        # 3. Generate Bridges (The Topology)
        # Interleaves connectivity tokens to create a continuous manifold.
        # Output: (Batch, 2*N - 1, N_Coeffs)
        chain = self._insert_bridges(data_tokens)

        return chain

    def output_processor(self, chain: torch.Tensor) -> torch.Tensor:
        """
        Spectral Chain -> Semantic Embedding/Logits.
        Usually called after the Routing Atom has collapsed the stack.
        """
        B, K, N = chain.shape
        return chain.view(B, -1)

    # --- INTERNAL HELPERS ---

    def _patchify(self, image: torch.Tensor) -> torch.Tensor:
        """Splits image into patch_size x patch_size blocks."""
        B, C, H, W = image.shape
        P = self.patch_size

        # Check divisibility (Production safeguard)
        if H % P != 0 or W % P != 0:
            raise ValueError(f"Image dims ({H},{W}) not divisible by patch size {P}")

        # Unfold: Extract sliding blocks
        # (B, C, H, W) -> (B, C, N_Rows, N_Cols, P, P) -> Flatten
        # PyTorch Unfold is the fastest way to do this
        unfolded = F.unfold(image, kernel_size=P, stride=P)
        # unfolded shape: (B, C*P*P, N_Patches)

        # Transpose to (B, N_Patches, Features)
        return unfolded.transpose(1, 2)

    def _insert_bridges(self, data_tokens: torch.Tensor) -> torch.Tensor:
        """
        Interleaves Bridge tokens between Data tokens.
        Logic: Bridge_i = 0.5 * (Data_i + Data_{i+1})
        """
        B, N, C = data_tokens.shape

        if N < 2:
            return data_tokens

        # 1. Identify Neighbors
        left = data_tokens[:, :-1, :]  # 0 to N-1
        right = data_tokens[:, 1:, :]  # 1 to N

        # 2. Create Bridges (Midpoint Manifold Interpolation)
        bridges = 0.5 * (left + right)

        # 3. Interleave (The Zipper)
        # We want: [L0, B0, L1, B1, L2, B2...] + [Last_Token]

        # Stack pairs: (B, N-1, 2, C)
        # This pairs (Data_i, Bridge_i) together
        paired = torch.stack([left, bridges], dim=2)

        # Flatten the pairs: (B, 2*(N-1), C)
        interleaved = paired.view(B, -1, C)

        # Append the final data token
        last_token = data_tokens[:, -1:, :] # (B, 1, C)
        chain = torch.cat([interleaved, last_token], dim=1)

        return chain

In [14]:
class TestInterface(unittest.TestCase):
    def setUp(self):
        self.B = 2
        self.C = 1 # Grayscale
        self.H, self.W = 8, 8 # Small image
        self.P = 4 # Patch Size
        self.N_Coeffs = G_CONFIG.n_coeffs # 8

        self.img = torch.randn(self.B, self.C, self.H, self.W)
        self.interface = Interface(patch_size=self.P, image_channels=self.C)

    def test_patch_shapes(self):
        """Verify 2D -> 1D serialization."""
        # 8x8 image with 4x4 patches = 4 patches total (2x2 grid)
        # Sequence Length should be 4 Data Tokens.

        patches = self.interface._patchify(self.img)
        expected_patches = 4

        self.assertEqual(patches.shape[1], expected_patches,
                         f"Patch count wrong. Got {patches.shape[1]}, expected 4")
        print("Test 1 (Patchify): Shape Validated")

    def test_bridge_topology(self):
        """Verify Bridge Insertion (Geometry)."""
        # Create dummy tokens: [0, 2, 4]
        # Bridges should be: [1, 3] (Averages)
        # Result Chain: [0, 1, 2, 3, 4]

        data = torch.zeros(1, 3, self.N_Coeffs)
        data[0, 0, :] = 0.0
        data[0, 1, :] = 2.0
        data[0, 2, :] = 4.0

        chain = self.interface._insert_bridges(data)

        # Check Length: 3 Data -> 2 Bridges -> Total 5
        self.assertEqual(chain.shape[1], 5, "Chain length mismatch")

        # Check Bridge Values
        # Index 1 is Bridge01 (Avg of 0 and 2 -> 1)
        # Index 3 is Bridge12 (Avg of 2 and 4 -> 3)
        bridge1 = chain[0, 1, 0].item()
        bridge2 = chain[0, 3, 0].item()

        self.assertAlmostEqual(bridge1, 1.0, places=4, msg="Bridge 1 value wrong")
        self.assertAlmostEqual(bridge2, 3.0, places=4, msg="Bridge 2 value wrong")

        print("Test 2 (Topology): Bridge Interpolation Validated")

    def test_full_pipeline(self):
        """Verify Full Input Processor."""
        # Image (8x8) -> 4 Patches
        # Chain Length = 2*4 - 1 = 7 Tokens
        chain = self.interface.input_processor(self.img)

        self.assertEqual(chain.shape[1], 7, "Full pipeline sequence length mismatch")
        self.assertEqual(chain.shape[2], self.N_Coeffs, "Coefficient dim mismatch")
        print("Test 3 (Pipeline): Full Flow Validated")

if __name__ == '__main__':
    suite = unittest.TestLoader().loadTestsFromTestCase(TestInterface)
    unittest.TextTestRunner(verbosity=0).run(suite)

----------------------------------------------------------------------
Ran 3 tests in 0.003s

OK


Test 2 (Topology): Bridge Interpolation Validated
Test 3 (Pipeline): Full Flow Validated
Test 1 (Patchify): Shape Validated



---
# 9. **`VisionSystem`** (The Assembly)
---

In [15]:
class VisionSystem(Module):
    """
    The Root Application.
    Standard Architecture: Interface -> Retina -> Cortex -> Router -> Logits.
    """
    def __init__(self,
                 image_size: int = 32,
                 patch_size: int = 4,
                 num_classes: int = 10,
                 depth: int = 3):
        super().__init__()

        # 1. The Translator
        self.interface = Interface(patch_size=patch_size)

        # 2. The Retina (Fine details, No reduction)
        # A Series of Filter Atoms that refine the signal locally.
        retina_layers = [
            Atom(AtomMode.FILTER, reduction_factor=1)
            for _ in range(depth)
        ]
        self.retina = Compound(retina_layers, CompoundMode.SERIES)

        # 3. The Cortex (Abstraction, Reduction)
        # A Series of Filter Atoms that merge tokens (Scalability).
        # We perform one heavy reduction (2x) here.
        cortex_layers = [
            Atom(AtomMode.FILTER, reduction_factor=2),
            Atom(AtomMode.FILTER, reduction_factor=1)
        ]
        self.cortex = Compound(cortex_layers, CompoundMode.SERIES)

        # 4. The Router (Decision Head)
        # Collapses the remaining chain into 'num_classes' concepts.
        self.router = Atom(AtomMode.ROUTING, query_bank_size=num_classes)

        # 5. Final Projection (Spectral -> Logits)
        # Projects the N coefficients of each class concept into a single scalar score.
        self.classifier = nn.Linear(G_CONFIG.n_coeffs, 1)

    def forward(self, image: torch.Tensor) -> torch.Tensor:
        # 1. Input Processing
        # (Batch, C, H, W) -> (Batch, Seq_Len, N_Coeffs)
        chain = self.interface.input_processor(image)

        # 2. Retina Processing (Local Physics)
        # Transforms signal, maintains topology
        chain = self.retina(chain)

        # 3. Cortex Processing (Topological Compression)
        # Merges tokens, reducing sequence length
        chain = self.cortex(chain)

        # 4. Routing (Global Awareness)
        # (Batch, Seq_Len, N) -> (Batch, Num_Classes, N)
        concepts = self.router(chain)

        # 5. Output Processing
        # (Batch, Num_Classes, N) -> (Batch, Num_Classes, 1) -> (Batch, Num_Classes)
        logits = self.classifier(concepts).squeeze(-1)

        return logits

In [16]:
class TestVisionSystem(unittest.TestCase):
    def setUp(self):
        # Config
        G_CONFIG.basis_degree = 7
        self.B = 2
        self.H, self.W = 32, 32
        self.Classes = 10
        self.Patch = 4

        # Instantiate System
        self.model = VisionSystem(
            image_size=self.H,
            patch_size=self.Patch,
            num_classes=self.Classes
        )

        # Fake Batch
        self.image = torch.randn(self.B, 1, self.H, self.W)

    def test_forward_pass(self):
        """Verify End-to-End Execution."""
        logits = self.model(self.image)

        # Check Shape: (Batch, Num_Classes)
        self.assertEqual(logits.shape, (self.B, self.Classes),
                         f"Output shape wrong. Got {logits.shape}, expected {(self.B, self.Classes)}")

        print("Test 1 (System): Forward Pass Validated")

    def test_gradient_flow(self):
        """Verify the Chain is unbroken (Trainable)."""
        logits = self.model(self.image)
        loss = logits.sum()
        loss.backward()

        # Check if gradients reached the Interface Projector (The very start)
        # If this is None, the graph is broken somewhere.
        grad_exists = self.model.interface.projector.weight.grad is not None
        self.assertTrue(grad_exists, "Gradient chain broken! Model cannot learn.")

        print("Test 2 (System): Gradient Flow Validated")

if __name__ == '__main__':
    suite = unittest.TestLoader().loadTestsFromTestCase(TestVisionSystem)
    unittest.TextTestRunner(verbosity=0).run(suite)

Test 1 (System): Forward Pass Validated


----------------------------------------------------------------------
Ran 2 tests in 0.087s

OK


Test 2 (System): Gradient Flow Validated


---
# temp
---

In [17]:
import torch
import torch.nn as nn
import torch.optim as optim

def run_simulation():
    print("========================================")
    print("      VISION SYSTEM: LIVE SIMULATION     ")
    print("========================================")

    # 1. SETUP: Hyperparameters
    # We use a small setup for quick debugging
    BATCH_SIZE = 4
    IMG_SIZE = 32
    PATCH_SIZE = 4
    NUM_CLASSES = 10
    DEPTH = 2           # Retina Depth
    DEGREE = 7          # Cheby Degree (N=8 coeffs)

    # Update Global Config
    Core.update_config(degree=DEGREE)
    device = Core.get_device()
    print(f"[*] Configuration: Image={IMG_SIZE}x{IMG_SIZE}, Patch={PATCH_SIZE}, Degree={DEGREE}")

    # 2. INSTANTIATE: The Model
    model = VisionSystem(
        image_size=IMG_SIZE,
        patch_size=PATCH_SIZE,
        num_classes=NUM_CLASSES,
        depth=DEPTH
    ).to(device)

    print(f"[*] Model Built. Parameters: {model.count_parameters():,}")

    # 3. DATA: Mock Inputs
    # Random noise mimicking a batch of grayscale images
    images = torch.randn(BATCH_SIZE, 1, IMG_SIZE, IMG_SIZE).to(device)
    # Random labels
    labels = torch.randint(0, NUM_CLASSES, (BATCH_SIZE,)).to(device)

    # 4. OPTIMIZER: Standard Adam
    optimizer = optim.Adam(model.parameters(), lr=0.001)
    criterion = nn.CrossEntropyLoss()

    print("\n[*] Starting Forward Pass...")

    # --- STEP A: FORWARD ---
    logits = model(images)

    print(f"    Output Shape: {logits.shape} (Expected: {BATCH_SIZE}, {NUM_CLASSES})")
    print(f"    Sample Logits (First Image): {logits[0].detach().cpu().numpy().round(2)}")

    # --- STEP B: LOSS CALCULATION ---
    loss = criterion(logits, labels)
    print(f"[*] Loss Calculated: {loss.item():.4f}")

    # --- STEP C: BACKWARD (Gradient Flow) ---
    optimizer.zero_grad()
    loss.backward()

    # Check gradients at the very beginning (Projector) and very end (Classifier)
    grad_start = model.interface.projector.weight.grad
    grad_end = model.classifier.weight.grad

    if grad_start is not None and grad_end is not None:
        grad_mag = grad_start.norm().item()
        print(f"[*] Backward Pass Successful.")
        print(f"    Gradient reached Input Projector (Magnitude: {grad_mag:.4f})")
    else:
        print("[!] FATAL: Gradient chain broken.")
        return

    # --- STEP D: OPTIMIZE ---
    optimizer.step()
    print("[*] Optimizer Step Complete. Weights updated.")

    print("\n========================================")
    print("      SIMULATION COMPLETE: SUCCESS      ")
    print("========================================")

# Run it
if __name__ == "__main__":
    run_simulation()

      VISION SYSTEM: LIVE SIMULATION     
[*] Configuration: Image=32x32, Patch=4, Degree=7
[*] Model Built. Parameters: 705

[*] Starting Forward Pass...
    Output Shape: torch.Size([4, 10]) (Expected: 4, 10)
    Sample Logits (First Image): [ 7.58  4.57  0.34 -0.02 11.84 11.84  7.63 11.84  6.29 11.84]
[*] Loss Calculated: 5.6603
[*] Backward Pass Successful.
    Gradient reached Input Projector (Magnitude: 16.8788)
[*] Optimizer Step Complete. Weights updated.

      SIMULATION COMPLETE: SUCCESS      
